<a href="https://colab.research.google.com/github/vitaliy887/IT-Cod_VKoltsov/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 7. Обработка данных из файла. Использование коллекций


ФИО студента: Кольцов Виталий Витальевич

Группа студента: ИНБ-б-о-26-2

Вариант: 1


## Задание


Файл `p7_data_01.csv` содержит информацию о сотрудниках:

- `ID` — уникальный ключ записи;
- `Fname`, `Lname` — имя и фамилия сотрудника;
- `Department` — категория (отдел: бухгалтеры, инженеры, программисты);
- `Money` — заработная плата в долларах;
- `HasDiploma` — признак наличия диплома (`true`/`false`).

Необходимо:

1. Найти количество сотрудников в каждой из 3-х категорий;
2. Найти суммарный фонд з/п и сумму по каждой категории;
3. Найти количество сотрудников с дипломом;
4. Найти максимальную зарплату среди сотрудников без диплома.


## Анализ задания, получение данных


Файл p7_data_01.csv расположен репозитории. Выполним копирование файла в локальное хранилище Google Colab, используя следующую команду:

In [2]:
!wget https://raw.githubusercontent.com/enikolaev/IT-and-Programming/refs/heads/main/data/pract7/p7_data_01.csv

--2026-11-16 17:39:59--  https://raw.githubusercontent.com/enikolaev/IT-and-Programming/refs/heads/main/data/pract7/p7_data_01.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63484 (62K) [text/plain]
Saving to: ‘p7_data_01.csv’

p7_data_01.csv      100%[===================>]  62.00K  --.-KB/s    in 0.009s  

2026-11-16 17:39:59 (6.81 MB/s) - ‘p7_data_01.csv’ saved [63484/63484]



Считаем файл как обычный текстовый файл: в цикле `for line in F` последовательно обходим все его строки, первую строку читаем методом `readline()` — это заголовки столбцов.


In [3]:
F=open('p7_data_01.csv')
n_to_print=5
for line in F:
  n_to_print-=1
  if n_to_print>0:
    print(line)


ID,Fname,Lname,Department,Money,HasDiploma

84-8725496,Christy,Darrigoe,бухгалтеры,$69537.84,false

43-7488475,Ulrich,Himsworth,инженеры,$46446.33,false

78-5637374,Arlena,Manktelow,программисты,$39916.04,false



Теперь разделим строки на отдельные поля методом `split(',')` и соберём список словарей `all_employees`. Значение зарплаты конвертируем в `float`, срезом убирая знак `$` с начала строки.


In [4]:
F=open('p7_data_01.csv')
header = F.readline().strip().split(',')
print(header)
print('-'*20)

# список всех сотрудников
all_employees=[]

for line in F:
  data = line.strip().split(',')
  emp = {} # создаем пустой словарь
  for i, key in enumerate(header): # обходим все заголовки
    emp[key] = data[i] # добавляем в словарь ключ:значение

  # конвертируем зарплату к числу, отбрасывая знак $
  emp['Money'] = float(emp['Money'][1:])

  all_employees.append(emp)

print(all_employees[:5]) # напечатаем первых 5 сотрудников

F.close() # файл больше не нужен


['ID', 'Fname', 'Lname', 'Department', 'Money', 'HasDiploma']
--------------------
[{'ID': '84-8725496', 'Fname': 'Christy', 'Lname': 'Darrigoe', 'Department': 'бухгалтеры', 'Money': 69537.84, 'HasDiploma': 'false'}, {'ID': '43-7488475', 'Fname': 'Ulrich', 'Lname': 'Himsworth', 'Department': 'инженеры', 'Money': 46446.33, 'HasDiploma': 'false'}, {'ID': '78-5637374', 'Fname': 'Arlena', 'Lname': 'Manktelow', 'Department': 'программисты', 'Money': 39916.04, 'HasDiploma': 'false'}, {'ID': '02-2512795', 'Fname': 'Karee', 'Lname': 'Merton', 'Department': 'бухгалтеры', 'Money': 2312.87, 'HasDiploma': 'false'}, {'ID': '18-5526133', 'Fname': 'Berte', 'Lname': 'McNess', 'Department': 'инженеры', 'Money': 78445.03, 'HasDiploma': 'true'}]


## Решение задачи 1. Количество сотрудников в каждой из 3-х категорий


Переберём всех сотрудников и соберём словарь счётчиков. Множество `categories` используем, чтобы показать уникальные категории в файле.


In [5]:
counts = {}
categories = set()
for emp in all_employees:
  cat = emp['Department']
  counts[cat] = counts.get(cat, 0) + 1
  categories.add(cat)

print('Уникальные категории:', categories)
print('Количество сотрудников по категориям:')
for cat, cnt in counts.items():
  print(cat, '-', cnt)


Уникальные категории: {'программисты', 'инженеры', 'бухгалтеры'}
Количество сотрудников по категориям:
бухгалтеры - 314
инженеры - 335
программисты - 351


## Решение задачи 2. Суммарный фонд з/п и суммы по каждой категории


In [6]:
total_money = 0
salaries = {}
for emp in all_employees:
  s = emp['Money']
  total_money += s
  salaries[emp['Department']] = salaries.get(emp['Department'], 0) + s

print('Суммарный фонд з/п: %.2f' % total_money)
for cat, s in salaries.items():
  print(cat, '- %.2f' % s)


Суммарный фонд з/п: 48992851.42
бухгалтеры - 15423509.92
инженеры - 17010539.27
программисты - 16558802.23


## Решение задачи 3. Количество сотрудников с дипломом


In [7]:
counter = 0
for emp in all_employees:
  if emp['HasDiploma'].lower() == 'true':
    counter += 1

print('Количество сотрудников с дипломом: ', counter)


Количество сотрудников с дипломом:  511


## Решение задачи 4. Максимальная зарплата среди сотрудников без диплома


In [8]:
max_salary = 0
target_emp = None

for emp in all_employees:
  if emp['HasDiploma'].lower() == 'false' and emp['Money'] > max_salary:
    max_salary = emp['Money']
    target_emp = emp

print('Максимальная зарплата без диплома: %.2f' % max_salary)
print('Сотрудник:', target_emp['Fname'], target_emp['Lname'],
      '(', target_emp['Department'], ',', target_emp['ID'], ')')


Максимальная зарплата без диплома: 99831.81
Сотрудник: Wendel Northway ( бухгалтеры , 74-7382174 )


В данной работе были использованы операторы `if` (условный оператор), `for ... in ...` (оператор цикла с обходом коллекции). Применялись функции `print()`, `open()`, `len()`. Использовались типы данных `list()`, `dict()`, `set()` и метод строк `split()`.


# Контрольные вопросы


###1.Сумма элементов списка p


In [9]:
p = [10, 100, 34, 56, 45, 45]

s = 0
for x in p:
  s += x
print(s)

# либо одной строкой
print(sum(p))


290
290


###2.Количество элементов списка p


In [10]:
p = [10, 100, 34, 56, 45, 45]
print(len(p))


6


###3.Сумма элементов, кратных 7


In [11]:
p = [10, 100, 35, 34, 56, 45, 45, 21]

s = 0
for x in p:
  if x % 7 == 0:
    s += x
print(s)


112


###4.Ключи словаря d, значения которых превышают 45


In [12]:
d = {4:10, 12:100, 35:3, 34:56, 56:11, 45:'abc'}

for k in d:
  if isinstance(d[k], (int, float)) and d[k] > 45:
    print(k)

# либо списком
print([k for k in d if isinstance(d[k], (int, float)) and d[k] > 45])


12
34
[12, 34]


Проверка `isinstance` нужна, так как значение по ключу 45 — строка `'abc'`, и сравнение строки с числом привело бы к ошибке.


###5.Список квадратов чисел от 1 до 100


In [13]:
squares = []
for i in range(1, 101):
  squares.append(i ** 2)

print(squares)


[1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324, 361, 400, 441, 484, 529, 576, 625, 676, 729, 784, 841, 900, 961, 1024, 1089, 1156, 1225, 1296, 1369, 1444, 1521, 1600, 1681, 1764, 1849, 1936, 2025, 2116, 2209, 2304, 2401, 2500, 2601, 2704, 2809, 2916, 3025, 3136, 3249, 3364, 3481, 3600, 3721, 3844, 3969, 4096, 4225, 4356, 4489, 4624, 4761, 4900, 5041, 5184, 5329, 5476, 5625, 5776, 5929, 6084, 6241, 6400, 6561, 6724, 6889, 7056, 7225, 7396, 7569, 7744, 7921, 8100, 8281, 8464, 8649, 8836, 9025, 9216, 9409, 9604, 9801, 10000]


###6.Структура файла csv и последовательность чтения


csv — это текстовый файл, в котором каждая строка является отдельной записью, а поля записи разделены разделителем (чаще всего запятой). Первая строка файла обычно содержит заголовки столбцов.

Последовательность действий при чтении:

1. Открыть файл функцией `open()`;
2. Прочитать первую строку (`readline()`), удалить концы строк методом `strip()` и разделить на заголовки `split(',')`;
3. Обходить файл в цикле `for line in F` — текстовый файл является коллекцией своих строк;
4. На каждом шаге `strip()` удаляет концы строк, а `split(',')` разрезает строку на отдельные поля;
5. Из полей собирается словарь (ключи — из заголовков, значения — из полей строки), при необходимости значения конвертируются к числам;
6. Словарь добавляется в список;
7. После чтения файл закрывается методом `close()`.


###7.Элемент словаря 'старт': 1472


Ключом является строка `'старт'`, значением — целое число `1472`.


###8.Инструкция employee['id'] = 54321


Инструкция добавляет в словарь `employee` пару `'id' : 54321`. Если ключ `'id'` уже существовал, его старое значение будет перезаписано новым.


###9.Что покажет фрагмент кода


In [14]:
stuff = {1:'ааа', 2:'ббб', 3: 'ввв'}
print(stuff[3])


ввв


Будет напечатано значение по ключу 3 — строка `ввв`.


###10.Как определить, существует ли пара ключ : значение в словаре


Наличие ключа проверяется оператором `in`: `key in d`. Для проверки конкретной пары ключ:значение удобно использовать метод `items()`: `(key, value) in d.items()`. Также можно применить метод `d.get(key)` — он вернёт `None`, если ключа нет, без ошибки `KeyError`.


###11.Инструкция del inventory[654]


Инструкция удаляет из словаря `inventory` пару с ключом `654`. Если такого ключа в словаре нет, произойдёт ошибка `KeyError`.


###12.Что покажет фрагмент кода


In [15]:
stuff = {1:'ааа', 2:'ббб', 3:'ввв'}
print(len(stuff))


3


Функция `len()` для словаря возвращает количество пар ключ:значение — будет напечатано `3`.


###13.Что покажет фрагмент кода


In [16]:
stuff = {1:'ааа', 2:'ббб', 3:'ввв'}
for k in stuff:
  print(k)


1
2
3


При обходе словаря циклом `for` перебираются его ключи, поэтому будут напечатаны `1`, `2`, `3`, каждый с новой строки.


###14.Разница между pop() и popitem()


Метод `pop(key)` удаляет из словаря пару по указанному ключу и возвращает её значение (можно задать значение по умолчанию: `d.pop(k, None)` — тогда ошибки `KeyError` не будет). Метод `popitem()` вызывается без аргументов: он удаляет последнюю добавленную пару и возвращает её в виде кортежа `(ключ, значение)`.


###15.Что возвращает метод items()


Метод `items()` возвращает представление словаря — набор кортежей `(ключ, значение)` всех его пар. Удобно использовать в цикле: `for k, v in d.items()`.


###16.Что возвращают методы keys() и values()


Метод `keys()` возвращает представление всех ключей словаря, а метод `values()` — представление всех значений. Для словаря выше это `[1, 2, 3]` и `['ааа', 'ббб', 'ввв']` соответственно.
